In [1]:
from collections import Counter
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [2]:
from collections import Counter
with open(r"dataset\tiny_shakespeare.txt","r",encoding="utf-8") as fp:
    text=fp.read()
lines=text.split("\n")
counter=Counter(text)
token=[]
words=[]    
for line in lines:
    words.extend(line.split())
    sequence=[]
    for word in line.split():
        sequence=list(word)+["<w>"]
        token.append(sequence)
#counter

In [3]:
def merge(common,token):
    i=0
    new_token=[]
    while i<len(token):
        if i<len(token)-1 and (token[i],token[i+1])==common:
            new_token.append("".join(common))
            i+=2
        else:
            new_token.append(token[i])
            i+=1
    return new_token

### Merging Frequent Charecters

In [4]:
for i in range(2):
    pairs=[]
    for i in token:
        for j in range(len(i)-1):
            pairs.append((i[j],i[j+1]))
    adjacent_frequency=Counter(pairs)
    print("\n\nFrequency of adjacent tokens:",adjacent_frequency)
    most_common=adjacent_frequency.most_common(1)[0][0]
    print("Most Common pair:",most_common)
    for i in range(len(token)):
        token[i]=merge(most_common,token[i])
print("\nFinal vocab tokens:",token)



Frequency of adjacent tokens: Counter({('e', '<w>'): 29077, ('t', 'h'): 22739, (',', '<w>'): 19603, ('h', 'e'): 18203, ('t', '<w>'): 17300, ('s', '<w>'): 16256, ('d', '<w>'): 14957, ('o', 'u'): 12730, ('e', 'r'): 11771, ('r', '<w>'): 10974, ('y', '<w>'): 10679, ('i', 'n'): 10606, (':', '<w>'): 10275, ('a', 'n'): 10197, ('r', 'e'): 9843, ('h', 'a'): 9673, ('n', 'd'): 9306, ('n', '<w>'): 9188, ('o', 'r'): 8458, ('a', 't'): 8339, ('o', '<w>'): 8284, ('.', '<w>'): 7811, ('e', 'n'): 7568, ('i', 's'): 7526, ('a', 'r'): 7081, ('h', 'i'): 6991, ('s', 't'): 6823, ('e', 's'): 6478, ('o', 'n'): 6435, ('l', 'l'): 6357, ('e', 'a'): 6288, ('m', 'e'): 6135, ('i', 't'): 6114, ('v', 'e'): 5986, ('t', 'o'): 5860, ('l', '<w>'): 5597, ('s', 'e'): 5507, ('h', '<w>'): 5328, ('u', 'r'): 5313, ('n', 'o'): 5209, ('h', 'o'): 5199, ('n', 'g'): 5140, ('f', '<w>'): 4817, ('y', 'o'): 4728, ('I', '<w>'): 4448, ('t', 'e'): 4433, ('l', 'e'): 4423, ('a', 'l'): 4149, ('o', 't'): 4060, ('o', 'w'): 3964, ('b', 'e'): 395

### Encoding

In [5]:
word2idx={}
idx2word={}
special_symbol=["<BOS>","<EOS>","<UNK>","<PAD>"]
vocab=sorted(set(words+special_symbol))
print("Length of the vocab:",len(vocab))
for idx,val in enumerate(vocab):
    word2idx[val]=idx
for val in special_symbol:
    if val not in word2idx:
        word2idx[val]=len(word2idx)
for idx,val in word2idx.items():
    idx2word[val]=idx

Length of the vocab: 25674


### Word to Token

In [6]:
batch=[]
for line in lines:
    new_token=["<BOS>"]+line.split()+["<EOS>"]
    token=[]
    for char in new_token:
        token.append(word2idx.get(char,word2idx["<UNK>"]))
    batch.append(token)

### Padding Tokens

In [7]:
maximum=max(map(lambda x:len(x),batch))
for i in range(len(batch)):
    EOS=batch[i].pop()
    for j in range(maximum-len(batch[i])-1):
        batch[i].append(word2idx["<PAD>"])
    batch[i].append(EOS)
print("The highest number of words in sentence:",maximum)
print("The number of lines: ",len(batch))

The highest number of words in sentence: 18
The number of lines:  40000


### Masked Padding

In [8]:
masked_padding=[[char!=word2idx["<PAD>"] for char in line] for line in batch ]

### Positional Encoding

In [9]:
class Encoding(nn.Module):
    def __init__(self,maximum_size,feature_size):
        super().__init__()
        encoding=torch.zeros(maximum_size,feature_size)
        print(encoding.shape)
        position=torch.arange(maximum_size).unsqueeze(1)
        div_term=torch.exp(torch.arange(0,feature_size,2)*(-torch.log(torch.tensor(10000.0))/feature_size))
        encoding[:,0::2]=torch.sin(position*div_term)
        encoding[:,1::2]=torch.cos(position*div_term)
        self.register_buffer("encoding",encoding)
    def forward(self,x):
        size=x.shape[1]
        return self.encoding[:size]+x

### Multi Head Attention

In [10]:
class Attention(nn.Module):
    def __init__(self,features,num_head):
        super().__init__()
        self.head_features=features//num_head
        self.num_head=num_head
        self.q=nn.Linear(features,features)
        self.k=nn.Linear(features,features)
        self.v=nn.Linear(features,features)
    def forward(self,x):
        q=self.q(x)
        k=self.k(x)
        v=self.v(x)
        q=q.view(q.size(0),q.size(1),self.num_head,self.head_features).transpose(1,2)
        k=k.view(k.size(0),k.size(1),self.num_head,self.head_features).transpose(1,2)
        v=v.view(v.size(0),v.size(1),self.num_head,self.head_features).transpose(1,2)
        score=q@k.transpose(-1,-2)/(self.head_features**0.5)
        mask=torch.tril(torch.ones(x.shape[1],x.shape[1])).to(x.device)
        score=score.masked_fill(mask==0,float("-inf"))
        scaled_score=score.softmax(dim=-1)
        attention=scaled_score@v
        attention=attention.transpose(1,2)
        attention=attention.contiguous().view(x.shape[0],x.shape[1],self.num_head*self.head_features)
        return attention

### Resnet

In [11]:
class Resnet_FNN(nn.Module):
    def __init__(self,features,num_head):
        super().__init__()
        self.attention=Attention(features,num_head)
        self.norm1=nn.LayerNorm(features)
        self.norm2=nn.LayerNorm(features)
        self.model=nn.Sequential(
            nn.Linear(features,features*4),
            nn.GELU(),
            nn.Linear(features*4,features)
        )  
    def forward(self,x):
        attention=self.attention(x)
        x=self.norm1(x+attention)
        ff=self.model(x)
        x=self.norm2(ff+x)
        return x

In [ ]:
features=10
num_heads=2
epoch=500
batch_size=1000

input_token=torch.tensor(batch,dtype=torch.long)
encoder=nn.Embedding(len(vocab),features).to(device)
positional_encoding=Encoding(maximum,features).to(device)

model=Resnet_FNN(features,num_heads)
model=model.to(device)
output_layer=nn.Linear(features,len(vocab)).to(device)
loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(list(encoder.parameters())+list(positional_encoding.parameters())+list(model.parameters())+list(output_layer.parameters()),lr=0.01)

x=input_token[:,:-1]
y=input_token[:,1:]
dataset=TensorDataset(x,y)
loader=DataLoader(dataset,batch_size=batch_size,shuffle=True)

total=0
for i in range(epoch):
    curr=0
    for x_batch,y_batch in loader:
        x_batch=x_batch.to(device)
        y_batch=y_batch.to(device)
        encoded_input=positional_encoding(encoder(x_batch)).to(device)
        output_token=input_token[:,1::1]
        optimizer.zero_grad()
        result=model(encoded_input)
        result=output_layer(result)
        loss=loss_fn(result.reshape(-1,len(vocab)),y_batch.reshape(-1))
        loss.backward()
        optimizer.step()
        curr+=loss.item()
    total=curr/len(loader)
    if i%10==0:
        print(f"epoch {i} : {total}")